# Titanic Dataset Analysis

This notebook performs the cleaning, feature engineering, and feature selection required for Artificial Intelligence Assignment 2.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT / "scripts"))
DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)

train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
train.head()

## Part 1: Data Cleaning

First we inspect missing values, then apply documented imputation and outlier handling decisions.

In [ ]:
train.isna().sum()

In [ ]:
from data_cleaning import clean_train

cleaned, cleaning_summary = clean_train(train)
cleaned.to_csv(DATA_DIR / "train_cleaned.csv", index=False)
print(cleaning_summary)
cleaned.head()

## Part 2: Feature Engineering

The engineered features capture family size, passenger title, cabin deck, age bands, and fare transformations.

In [ ]:
from feature_engineering import engineer_features

featured = engineer_features(cleaned)
featured.to_csv(DATA_DIR / "train_features.csv", index=False)
featured.head()

In [ ]:
cleaned["Fare"].hist(bins=30)
plt.title("Fare Before Log Transform")
plt.xlabel("Fare")
plt.ylabel("Passengers")
plt.show()

In [ ]:
featured["LogFare"].hist(bins=30)
plt.title("Fare After Log Transform")
plt.xlabel("LogFare")
plt.ylabel("Passengers")
plt.show()

In [ ]:
cleaned.groupby("Sex")["Survived"].mean().plot(kind="bar")
plt.title("Survival Rate by Sex")
plt.ylabel("Survival rate")
plt.show()

In [ ]:
cleaned.groupby("Pclass")["Survived"].mean().plot(kind="bar")
plt.title("Survival Rate by Passenger Class")
plt.ylabel("Survival rate")
plt.show()

## Part 3: Feature Selection

Feature ranking uses Random Forest when scikit-learn is available. If it is not installed, the script falls back to absolute Pearson correlation so the workflow remains reproducible.

In [ ]:
from feature_selection import numeric_model_frame, highly_correlated_features, random_forest_importance, pearson_importance, select_features

x, y = numeric_model_frame(featured)
redundant = highly_correlated_features(x)
importance = random_forest_importance(x, y)
if importance is None:
    importance = pearson_importance(x, y)
selected = select_features(importance, redundant)
importance.head(15)

In [ ]:
selected